## Computational Physics -- Problem Set 1
##### _Theodore Landa_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.
For this problem, I used visualizations (figures 1-4) to zone in on the regions where the relevant numerics break. The visualizations don't look great, but they're not the point of this problem!

_(a)_ The smallest $\delta$ we can add is just over $1.11\times 10^{-16}$ for the 64-bit representation (double?).

_(b)_ Using the python 64-bit representation, we get that the largest $\Delta$ is just over $1.8\times 10^{16}$ (though now it's messy, as seen in the graph).

_(c)_ The largest $\Delta_1$ is about 745. The corresponding $\exp(-\Delta_1)$ is $5\times 10^{-324}$. The largest $\Delta_2$ is about 709.7, and the corresponding $\exp(\Delta_2)$ is about $1.8 \times 10^{308}$.

In [ ]:
x = np.logspace(-16, -15.9, 1000)
y = x + np.float64(1.0)
test = y>1.0
test = test.astype(int)
plt.semilogx(x, test)
plt.xlabel(r'$\delta$')
plt.ylabel('boolean pass')
plt.title(r'$[1+\delta]>1$ ?')
lowest = np.argmax(test)

print(x[lowest])
print(x[lowest-1])
plt.savefig('fig1.svg')

In [ ]:
x = np.logspace(16.23, 16.28, 1000)
y = x + np.float64(1.0)
test = y>x
test = test.astype(int)
plt.semilogx(x, test)
plt.xlabel(r'$\Delta$')
plt.ylabel('boolean pass')
plt.title(r'$[1+\Delta]>\Delta$ ?')
lowest = np.argmax(test[::-1])
print(x[1000-lowest])
print(x[1000-lowest+1])
plt.savefig('fig2.svg')

In [ ]:
x = np.logspace(2.7, 3, 1000)
y = np.exp(-1*x)
test = y>0
test = test.astype(int)
plt.semilogx(x, test)
plt.xlabel(r'$\Delta_1$')
plt.ylabel('boolean pass')
plt.title(r'$\exp(-\Delta_1)>0$ ?')
lowest = np.argmax(test[::-1])
print(x[1000-lowest-1])
print(y[1000-lowest-1])
plt.savefig('fig3.svg')

In [ ]:
x = np.logspace(2.84, 2.87, 1000)
y = np.exp(x)
test = y != np.inf
test = test.astype(int)
plt.semilogx(x, test)
plt.xlabel(r'$\Delta_2$')
plt.ylabel('boolean pass')
plt.title(r'$\exp(\Delta_2)$ float?')
lowest = np.argmax(test[::-1])
print(x[1000-lowest-1])
print(y[1000-lowest-1])
plt.savefig('fig4.svg')

### Problem 2.
_(a)_ See figure 5. The scale height of the atmosphere (10 km) might be relevant because it allows us to infer whether or not a muon came from space based on its speed / energy!?

_(b)_ I addressed parametrization of speeds as follows: for the linear scale plots I instantiated a linearly spaced array of gamma values, and then computed the corresponding beta values. For the log scale plots, I instantiated a logarithmically spaced array of gamma values, which was great because I got a relatively high density of points for the curved part of the figure without having an extremely high resolution for the linear part. I also ran into issues with the graph vs $1-\beta$, because of catastrophic subtraction at high relativistic speeds. This was resolved by computing $1-\beta$ as $$1-\beta=\frac{1-\beta^2}{1+\beta}=\frac{1}{\gamma^2\,(1+\beta)}.$$ Here, even if the computer (floating point) can't tell the difference between $\beta$ and $1$, we still get an appropriately low number by using division instead of subtraction.  

In [ ]:
lifetime = 2.2*10**(-6)
c = 3*10**8
res = 10**6
gamma = np.linspace(1, 40, res)
beta = np.sqrt(1 - 1/gamma**2)
d = c*lifetime*np.sqrt(gamma**2 - 1)
naive = c*lifetime
atm = 10000

fig, axs = plt.subplots(2, 2, layout = 'constrained', figsize = (9, 7))
fig.suptitle('On My Muon Shit')

axs[0, 0].plot(beta, d, 'k-', lw = 3, label = 'distance travelled by muon')
axs[0, 0].axhline(naive, c = 'green', ls = '--', lw = 2, label = 'naive maximum distance')
axs[0, 0].axhline(atm, c = 'deepskyblue', ls = '--', lw = 2, label = 'height of atmosphere')
axs[0, 0].plot([], [], c = 'orangered', ls = '--', lw = 2, label = 'solar distance')
axs[0, 0].set_title(r'$\beta$ vs Distance')
axs[0, 0].set(xlabel = r"$\beta$ [dimensionless]")
axs[0, 0].set(ylabel = 'Distance [m]')
axs[0, 0].legend()

axs[0, 1].plot(gamma, d, 'k-', lw = 3)
axs[0, 1].axhline(naive, c = 'green', ls = '--', lw = 2)
axs[0, 1].axhline(atm, c = 'deepskyblue', ls = '--', lw = 2)
axs[0, 1].set_title(r'$\gamma$ vs Distance')
axs[0, 1].set(xlabel = r"$\gamma$ [dimensionless]")
axs[0, 1].set(ylabel = 'Distance [m]')

solardist = 150*10**9
gamma = np.logspace(0, 9.5, res)
beta = np.sqrt(1 - 1/gamma**2)
minusbeta = 1 / (gamma**2*(1+beta))
d = c*lifetime*np.sqrt(gamma**2 - 1)

axs[1, 0].set(xlabel = r"$1-\beta$ [dimensionless]")
axs[1, 0].set(ylabel = 'Distance [m]')
axs[1, 0].set_title(r'$1-\beta$ vs Distance, log-log')
axs[1, 0].loglog(minusbeta, d, 'k-', lw = 3, label = 'distance travelled by muon')
axs[1, 0].axhline(naive, c = 'green', ls = '--', lw = 2, label = 'naive maximum distance')
axs[1, 0].axhline(atm, c = 'deepskyblue', ls = '--', lw = 2, label = 'height of atmosphere')
axs[1, 0].axhline(solardist, c = 'orangered', ls = '--', lw = 2, label = 'solar distance')

axs[1, 1].set(xlabel = r"$\gamma$ [dimensionless]")
axs[1, 1].set(ylabel = 'Distance [m]')
axs[1, 1].set_title(r'$\gamma$ vs Distance, log-log')
axs[1, 1].loglog(gamma, d, 'k-', lw = 3, label = 'distance travelled by muon')
axs[1, 1].axhline(naive, c = 'green', ls = '--', lw = 2, label = 'naive maximum distance')
axs[1, 1].axhline(atm, c = 'deepskyblue', ls = '--', lw = 2, label = 'height of atmosphere')
axs[1, 1].axhline(solardist, c = 'orangered', ls = '--', lw = 2, label = 'solar distance')

plt.savefig('fig5.svg')
plt.show()

### Problem 3.
_(a)_ See figure 6. I _think_ I got the edges of the pixels right. My strategy was to make my x and y value arrays the midpoints of what I want each pixel to be, and call the function on that. Then, I rescaled the axes, and plotted the extrema lines according to their positions on those axes (and they match up with the colormap extrema, which are given by my x, y array scheme!). 

_(b)_ See figure 7. The peaks don't really get sharper; they just get more localized. We can't really tell if they're precisely located because the resolution is pretty low.

_(c)_ See figures 8 and 9. I feel good about my plots!

In [ ]:
def f(theta, alpha):
    return np.sin(2*theta + 1) * np.sin(3*alpha - np.pi/4)

def makecmap(power, xnumpix, ynumpix):
    fig, ax = plt.subplots()
    pixelx, pixely = np.meshgrid(np.linspace((np.pi / (2*xnumpix)), 2*np.pi - (np.pi / (2*xnumpix)), xnumpix), np.linspace((np.pi / (2*ynumpix)), 2*np.pi - (np.pi / (2*ynumpix)), ynumpix))
    im = ax.imshow(f(pixelx, pixely)**power, cmap = 'viridis', interpolation = 'nearest', origin = 'lower', extent = (0, 2*np.pi, 0, 2*np.pi))

    for i in [1, 3, 5, 7]:
        plt.axvline((i*np.pi/2 - 1)/2, c = 'r')

    for i in [1, 3, 5, 7, 9, 11]:
        plt.axhline((i*np.pi/2 + (np.pi/4))/3, c = 'r')
    
    ax.set(xlabel = r'$\theta$ [radians]', ylabel = r'$\alpha$ [radians]')
    
    if (power == 1):
        ax.figure.colorbar(im, label = r'$f\,(\theta, \alpha)$')
        ax.set(title = r'$f\,(\theta, \alpha)$ on '+str(ynumpix)+'x'+str(xnumpix)+' pixels')

    if (power == 5):
        ax.figure.colorbar(im, label = r'$f\,(\theta, \alpha)^5$')
        ax.set(title = r'$f\,(\theta, \alpha)^5$ on '+str(ynumpix)+'x'+str(xnumpix)+' pixels')
    

makecmap(1, 18, 30)
plt.savefig('fig6.svg')

In [ ]:
makecmap(5, 18, 30)
plt.savefig('fig7.svg')

In [ ]:
makecmap(1, 521, 601)
plt.savefig('fig8.svg')

In [ ]:
makecmap(5, 521, 601)
plt.savefig('fig9.svg')

### Problem 4.
We perform four tests (figures 10-13): of the pythagorean identity, of the odd parity of $\sin$, of the double angle identity for $\sin$, and of the complementarity of $\cos$ to $\sin$ (that $\cos$ is the $\sin$ of the complement). For the pythagorean identity, we can see that errors appear at all scales, at roughly the same magnitude ($10^{-16}$, where float stops being precise as we know from earlier). We see no errors in the parity of $\sin$, which makes sense if that's built into the numpy $\sin$ function. For the latter two tests, I've also plotted log-log plots, because I think they're informative. For the double angle formula and the complementarity test, we see that the error increases with scale, and the log-log tells us how more precisely; in both graphs there's an interesting step-function-like behavior that seems to indicate some kind of discretized power law for the error. In the double angle test, the error tops out at order $10^{-16}$, but the error in the complementarity test seems to just increase, which is probably because there's addition/subtraction involved between a big big number and a small number!

In [ ]:
def makedubviz(x, d, test):
    fig, axs = plt.subplots(1, 2, layout = 'constrained', figsize = (10, 3))
    axs[0].semilogx(x, d)
    axs[1].loglog(x, d)
    fig.suptitle(test + ' Residual')
    axs[0].set(title = 'semi-log', xlabel = r'$\theta$ [radians]', ylabel = r'$\delta$')
    axs[1].set(title = 'log-log', xlabel = r'$\theta$ [radians]', ylabel = r'$\delta$')

def makeviz(x, d, test):
    fig, axs = plt.subplots()
    axs.semilogx(x, d)
    fig.suptitle(test + ' Residual')
    axs.set(title = 'semi-log', xlabel = r'$\theta$ [radians]', ylabel = r'$\delta$')

x = np.logspace(-4, 4, 2000)
s = np.sin(x)
c = np.cos(x)
d1 = s**2 + c**2 - 1
makeviz(x, d1, r'$\delta = \cos^2(\theta)+\sin^2(\theta)-1$')
plt.savefig('fig10.svg')

In [ ]:
d2 = np.sin(x)+np.sin(-1*x)
makeviz(x, d2, r'$\delta = \sin(\theta)+\sin(-\theta)$')
plt.savefig('fig11.svg')

In [ ]:
d3 = np.sin(2*x)-2*s*c
makedubviz(x, d3, r'$\delta = \sin(2\theta)-2\sin(\theta)\,\cos(\theta)$')
plt.savefig('fig12.svg')

In [ ]:
d4 = np.sin(x)-np.cos(x-np.pi/2)
makedubviz(x, d4, r'$\delta = \sin(\theta)-\cos(\theta-\pi / 2)$')
plt.savefig('fig13.svg')

### Credits: 
Gabe Rosen-Turits and Nana Miller for some tips with graphing problem 2, Newman Computational Physics for saying slightly useful things about density plots, Matplotlib documentation for basically everything, chatGPT for helping me figure out how to avoid the numerical error in problem 2 by usefully rewriting $1-\beta$ as explained.